# Diabetes Prediction Using Machine Learning



## 1. Import Required Libraries



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

try:
    import matplotlib.pyplot as plt
except ImportError as error:
    plt = None
    print(f"matplotlib is not installed yet: {error}")

try:
    import seaborn as sns
except ImportError as error:
    sns = None
    print(f"seaborn is not installed yet: {error}")

try:
    import sklearn
except ImportError as error:
    sklearn = None
    print(f"scikit-learn is not installed yet: {error}")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 2. Define Dataset Path




In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks" and not (PROJECT_ROOT / "data" / "raw" / "diabetes.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_RELATIVE_PATH = Path("data") / "raw" / "diabetes.csv"
OUTPUT_TABLES_RELATIVE_DIR = Path("outputs") / "tables" / "week_04"

DATASET_PATH = PROJECT_ROOT / DATASET_RELATIVE_PATH
OUTPUT_TABLES_DIR = PROJECT_ROOT / OUTPUT_TABLES_RELATIVE_DIR

print("Project root: .")
print(f"Dataset path: {DATASET_RELATIVE_PATH}")


Project root: .
Dataset path: data\raw\diabetes.csv


## 3. Check Dataset File Exists




In [19]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset file not found: {DATASET_PATH}")

print("Dataset file found successfully.")


Dataset file found successfully.


## 4. Load Dataset




In [3]:
EXPECTED_COLUMNS = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome",
]

df = pd.read_csv(DATASET_PATH)

missing_columns = [column for column in EXPECTED_COLUMNS if column not in df.columns]
extra_columns = [column for column in df.columns if column not in EXPECTED_COLUMNS]

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

if extra_columns:
    print(f"Extra columns found in dataset: {extra_columns}")

print("Dataset loaded successfully.")


Dataset loaded successfully.


## 5. First and Last Five Records




In [4]:
print("First five records:")
display(df.head())

print("Last five records:")
display(df.tail())


First five records:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


Last five records:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1
767,1,93,70,31,0,30.4,0.315,23,0


## 6. Dataset Shape



In [5]:
dataset_shape = df.shape
print(f"Dataset shape: {dataset_shape}")


Dataset shape: (768, 9)


## 7. Column Names




In [6]:
column_names = list(df.columns)
column_names


['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age',
 'Outcome']

## 8. Data Types




In [7]:
data_types = df.dtypes.reset_index()
data_types.columns = ["column_name", "data_type"]
display(data_types)


,column_name,data_type
0,Pregnancies,int64
1,Glucose,int64
2,BloodPressure,int64
3,SkinThickness,int64
4,Insulin,int64
5,BMI,float64
6,DiabetesPedigreeFunction,float64
7,Age,int64
8,Outcome,int64


## 9. Number of Rows and Columns




In [8]:
number_of_rows = df.shape[0]
number_of_columns = df.shape[1]

print(f"Number of rows: {number_of_rows}")
print(f"Number of columns: {number_of_columns}")


Number of rows: 768
Number of columns: 9


## 10. Input Features and Target Variable



In [9]:
target_variable = "Outcome"
input_features = [column for column in df.columns if column != target_variable]

print("Input features:")
for feature in input_features:
    print(f"- {feature}")

print(f"Target variable: {target_variable}")


Input features:
- Pregnancies
- Glucose
- BloodPressure
- SkinThickness
- Insulin
- BMI
- DiabetesPedigreeFunction
- Age
Target variable: Outcome


## 11. Descriptive Statistics




In [10]:
descriptive_stats = df.describe().T
display(descriptive_stats)


,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00


## 12. Duplicate Records



In [11]:
duplicate_count = int(df.duplicated().sum())
print(f"Duplicate records: {duplicate_count}")


Duplicate records: 0


## 13. Explicit Missing Values




In [12]:
missing_values = pd.DataFrame({
    "column_name": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (df.isna().mean().values * 100).round(2),
})

display(missing_values)


,column_name,missing_count,missing_percentage
0,Pregnancies,0,0.0
1,Glucose,0,0.0
2,BloodPressure,0,0.0
3,SkinThickness,0,0.0
4,Insulin,0,0.0
5,BMI,0,0.0
6,DiabetesPedigreeFunction,0,0.0
7,Age,0,0.0
8,Outcome,0,0.0


## 14. Target-Class Counts and Percentages




In [13]:
target_distribution = (
    df[target_variable]
    .value_counts()
    .sort_index()
    .rename_axis(target_variable)
    .reset_index(name="count")
)
target_distribution["percentage"] = (target_distribution["count"] / len(df) * 100).round(2)

display(target_distribution)


,Outcome,count,percentage
0,0,500,65.1
1,1,268,34.9


## 15. Feature Documentation Table




In [14]:
feature_documentation = pd.DataFrame([
    {"feature_name": "Pregnancies", "meaning": "Number of times the patient has been pregnant", "role": "Input feature", "expected_data_type": "Integer", "expected_unit": "Count"},
    {"feature_name": "Glucose", "meaning": "Plasma glucose concentration measured during a glucose tolerance test", "role": "Input feature", "expected_data_type": "Integer or float", "expected_unit": "mg/dL"},
    {"feature_name": "BloodPressure", "meaning": "Diastolic blood pressure", "role": "Input feature", "expected_data_type": "Integer or float", "expected_unit": "mm Hg"},
    {"feature_name": "SkinThickness", "meaning": "Triceps skin fold thickness", "role": "Input feature", "expected_data_type": "Integer or float", "expected_unit": "mm"},
    {"feature_name": "Insulin", "meaning": "Two-hour serum insulin level", "role": "Input feature", "expected_data_type": "Integer or float", "expected_unit": "mu U/mL"},
    {"feature_name": "BMI", "meaning": "Body mass index", "role": "Input feature", "expected_data_type": "Float", "expected_unit": "kg/m^2"},
    {"feature_name": "DiabetesPedigreeFunction", "meaning": "Score representing diabetes family history influence", "role": "Input feature", "expected_data_type": "Float", "expected_unit": "Unitless score"},
    {"feature_name": "Age", "meaning": "Age of the patient", "role": "Input feature", "expected_data_type": "Integer", "expected_unit": "Years"},
    {"feature_name": "Outcome", "meaning": "Diabetes diagnosis class where 0 means non-diabetic and 1 means diabetic", "role": "Target variable", "expected_data_type": "Integer category", "expected_unit": "0 or 1 class label"},
])

display(feature_documentation)


,feature_name,meaning,role,expected_data_type,expected_unit
0,Pregnancies,Number of times the patient has been pregnant,Input feature,Integer,Count
1,Glucose,Plasma glucose concentration measured during a...,Input feature,Integer or float,mg/dL
2,BloodPressure,Diastolic blood pressure,Input feature,Integer or float,mm Hg
3,SkinThickness,Triceps skin fold thickness,Input feature,Integer or float,mm
4,Insulin,Two-hour serum insulin level,Input feature,Integer or float,mu U/mL
5,BMI,Body mass index,Input feature,Float,kg/m^2
6,DiabetesPedigreeFunction,Score representing diabetes family history inf...,Input feature,Float,Unitless score
7,Age,Age of the patient,Input feature,Integer,Years
8,Outcome,Diabetes diagnosis class where 0 means non-dia...,Target variable,Integer category,0 or 1 class label


## 16. Initial Observations

- The dataset contains 768 records and 9 columns.
- There are 8 input features and one target variable named `Outcome`.
- All columns are numeric in the loaded CSV file.
- The dataset has 0 duplicate records.
- The dataset has 0 explicit missing values.
- Target class `0` has 500 records (65.1%), and target class `1` has 268 records (34.9%).
- Some medical columns may require deeper checking in later weeks because zero can be unusual for measurements such as glucose, blood pressure, insulin, skin thickness, or BMI.


## 17. Save Summary Tables

Useful Week 4 summary tables are saved in `outputs/tables/`. These files are created from the raw dataset without modifying the original CSV.


In [15]:
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

dataset_overview = pd.DataFrame({
    "item": [
        "dataset_path",
        "number_of_rows",
        "number_of_columns",
        "duplicate_records",
        "explicit_missing_values",
        "input_features_count",
        "target_variable",
    ],
    "value": [
        str(DATASET_PATH.relative_to(PROJECT_ROOT)),
        number_of_rows,
        number_of_columns,
        duplicate_count,
        int(df.isna().sum().sum()),
        len(input_features),
        target_variable,
    ],
})

column_info = data_types.copy()
descriptive_stats_to_save = descriptive_stats.reset_index().rename(columns={"index": "column_name"})
duplicate_summary = pd.DataFrame({"check": ["duplicate_records"], "count": [duplicate_count]})

tables_to_save = {
    "dataset_overview.csv": dataset_overview,
    "column_info.csv": column_info,
    "descriptive_statistics.csv": descriptive_stats_to_save,
    "duplicate_summary.csv": duplicate_summary,
    "missing_values.csv": missing_values,
    "target_distribution.csv": target_distribution,
    "feature_documentation.csv": feature_documentation,
}

for file_name, table in tables_to_save.items():
    table.to_csv(OUTPUT_TABLES_DIR / file_name, index=False)

print("Saved summary tables:")
for file_name in tables_to_save:
    print(f"- {OUTPUT_TABLES_RELATIVE_DIR / file_name}")


Saved summary tables:
- outputs\tables\week_04\dataset_overview.csv
- outputs\tables\week_04\column_info.csv
- outputs\tables\week_04\descriptive_statistics.csv
- outputs\tables\week_04\duplicate_summary.csv
- outputs\tables\week_04\missing_values.csv
- outputs\tables\week_04\target_distribution.csv
- outputs\tables\week_04\feature_documentation.csv
